In [ ]:
import pandas as pd
import numpy as np
import os

DATA_FOLDER  = "data/raw"
CLEAN_FOLDER = "data/cleaned" 

os.makedirs(CLEAN_FOLDER, exist_ok=True)

pd.set_option("display.max_columns", None)
print("Setup done")


In [ ]:
files = {
    "sessions":           "website_sessions.csv",
    "pageviews":          "website_pageviews.csv",
    "orders":             "orders.csv",
    "order_items":        "order_items.csv",
    "order_item_refunds": "order_item_refunds.csv",
    "products":           "products.csv",
}
tables = {}
for name, file in files.items():
    tables[name] = pd.read_csv(os.path.join(DATA_FOLDER, file))
    print(name, "loaded:", tables[name].shape[0], "rows,", tables[name].shape[1], "columns")

In [ ]:
for name, df in tables.items():
    print("=" * 60)
    print(name.upper(), "->", df.shape)
    print(df.dtypes)
    print()

tables["sessions"].head()

In [ ]:
for name, df in tables.items():
    if "created_at" in df.columns:
        df["created_at"] = pd.to_datetime(df["created_at"])
        print(name, "-> created_at is now", df["created_at"].dtype)

In [ ]:
for name, df in tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]         
    if len(missing) == 0:
        print(name, "-> no missing values")
    else:
        print(name, "-> missing values found:")
        print(missing)
        print()

In [ ]:
primary_keys = {
    "sessions":           "website_session_id",
    "pageviews":          "website_pageview_id",
    "orders":             "order_id",
    "order_items":        "order_item_id",
    "order_item_refunds": "order_item_refund_id",
    "products":           "product_id",
}

for name, key in primary_keys.items():
    df = tables[name]
    dup_ids  = df[key].duplicated().sum()
    dup_rows = df.duplicated().sum()
    print(name, "-> duplicate IDs:", dup_ids, "| duplicate rows:", dup_rows)
    
    if dup_rows > 0:                          # only drop if we really found duplicates
        tables[name] = df.drop_duplicates()
        print("   removed", dup_rows, "duplicate rows")

In [ ]:
for name, df in tables.items():
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):     
            df[col] = df[col].str.strip()

sessions = tables["sessions"]
for col in ["utm_source", "utm_campaign", "device_type"]:
    sessions[col] = sessions[col].str.lower()

for col in ["utm_source", "utm_campaign", "device_type"]:
    print(sessions[col].value_counts(dropna=False))
    print()


In [ ]:
conditions = [
    sessions["utm_source"].notna(),        
    sessions["http_referer"].notna(),
]
choices = [
    sessions["utm_source"],
    "organic_search",
]

sessions["channel"] = np.select(conditions, choices, default="direct")

sessions["campaign"] = sessions["utm_campaign"].fillna("none")

for col in ["utm_source", "utm_campaign", "utm_content", "http_referer"]:
    sessions[col] = sessions[col].fillna("none")
    
print(sessions["channel"].value_counts())
print()
print(sessions.groupby(["channel", "campaign"]).size())


In [ ]:
for name in ["sessions", "orders", "order_items", "order_item_refunds"]:
    df = tables[name]
    df["date"]       = df["created_at"].dt.date                 
    df["year"]       = df["created_at"].dt.year                 
    df["month"]      = df["created_at"].dt.month               
    df["year_month"] = df["created_at"].dt.strftime("%Y-%m")    

print(tables["orders"][["created_at", "date", "year", "month", "year_month"]].head())


In [ ]:
# Add a profit column
# Gross profit = price − cost of goods sold.
orders      = tables["orders"]
order_items = tables["order_items"]

orders["gross_profit_usd"]      = (orders["price_usd"] - orders["cogs_usd"]).round(2)
order_items["gross_profit_usd"] = (order_items["price_usd"] - order_items["cogs_usd"]).round(2)

print(orders[["price_usd", "cogs_usd", "gross_profit_usd"]].head())

In [ ]:
print(orders[["price_usd", "cogs_usd", "gross_profit_usd"]].describe().round(2))
print()
print("Orders with price <= 0 :", (orders["price_usd"] <= 0).sum())
print("Orders with negative profit :", (orders["gross_profit_usd"] < 0).sum())

In [ ]:
links = [
    # (child table, key column, parent table)
    ("pageviews",          "website_session_id", "sessions"),
    ("orders",             "website_session_id", "sessions"),
    ("order_items",        "order_id",           "orders"),
    ("order_item_refunds", "order_item_id",      "order_items"),
    ("order_items",        "product_id",         "products"),
]
for child, key, parent in links:
    orphans = (~tables[child][key].isin(tables[parent][key])).sum()   # same column name in both tables
    print(child, "->", parent, "| rows with no match:", orphans)


In [ ]:
total_sessions = len(tables["sessions"])
total_orders   = len(tables["orders"])
total_revenue  = tables["orders"]["price_usd"].sum()

print("Total sessions          :", total_sessions)
print("Total orders            :", total_orders)
print("Conversion rate         :", round(total_orders / total_sessions * 100, 2), "%")
print("Total revenue           : $", round(total_revenue, 2))
print("Revenue per order (AOV) : $", round(total_revenue / total_orders, 2))
print("Revenue per session     : $", round(total_revenue / total_sessions, 4))
print()

# by year (2012 and 2015 are partial years: data runs 19 Mar 2012 to 19 Mar 2015)
by_year = pd.DataFrame({
    "sessions": tables["sessions"].groupby("year").size(),
    "orders":   tables["orders"].groupby("year").size(),
    "revenue":  tables["orders"].groupby("year")["price_usd"].sum().round(2),
})
by_year["conversion_%"] = (by_year["orders"] / by_year["sessions"] * 100).round(2)
by_year


In [ ]:
for name, file in files.items():
    out_path = os.path.join(CLEAN_FOLDER, "clean_" + file)
    tables[name].to_csv(out_path, index=False)
    print("saved:", out_path, "|", tables[name].shape[0], "rows,", tables[name].shape[1], "columns")

In [ ]:
check = pd.read_csv(os.path.join(CLEAN_FOLDER, "clean_website_sessions.csv"))
print(check.shape)
check.head()

In [ ]:
import pandas as pd

df = pd.read_csv("data/cleaned/clean_website_sessions.csv")
rest = df[df["website_session_id"] > 436089]
print(len(rest))          # should print 36782
rest.to_csv("data/cleaned/clean_website_sessions_rest.csv", index=False)